# 12. NLP와 트랜스포머 기초

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/%EC%8B%A4%EC%8A%B5_12_NLP%EC%99%80_%ED%8A%B8%EB%9E%9C%EC%8A%A4%ED%8F%AC%EB%A8%B8_%EA%B8%B0%EC%B4%88.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## 텍스트 전처리와 토큰화

### 🔬 실습 — 국방 도메인 토크나이저 만들기: 일반 토크나이저와의 차이를 눈으로

① **일반 토크나이저로 군사 용어를 쪼개 보기**


In [ ]:
from transformers import AutoTokenizer

tok_general = AutoTokenizer.from_pretrained("klue/bert-base")
for term in ["대전차화기", "K-9 자주포", "GOP 경계작전", "TOD 운용"]:
    print(f"{term:12s} -> {tok_general.tokenize(term)}")

**실행 결과 예시**

```
대전차화기        -> ['대전', '##차', '##화기']
K-9 자주포      -> ['K', '-', '9', '자주', '##포']
GOP 경계작전     -> ['GO', '##P', '경계', '##작전']
TOD 운용       -> ['TO', '##D', '운용']
```

대전차화기는 세 조각, K-9 자주포는 다섯 조각이 났다. 일반 코퍼스에 드문 표현이라 어휘에 그 조합이 없어서다.


② **도메인 코퍼스로 BPE 학습**: 군사 문서 텍스트 파일(예: 공개 교범과 뉴스)로 어휘를 새로 만든다.


In [ ]:
import os
# 도메인 코퍼스 준비: 실제 문서 대신 소규모 합성 군사 코퍼스를 파일로 만든다.
# (Colab 자립 실행용 — 실제로는 공개 교범·뉴스 등 대량 텍스트를 사용한다.)
mil_sentences = [
    "대전차 화기 운용 절차를 숙지한다",
    "대전차 화기 사격 훈련을 실시한다",
    "K-9 자주포 사격 훈련 일정을 확인한다",
    "K-9 자주포 정비 주기를 점검한다",
    "GOP 경계작전 상황을 보고한다",
    "GOP 경계작전 근무 규칙을 준수한다",
    "TOD 운용 요원이 야간 감시를 수행한다",
    "전방 관측소에서 적 병력 이동을 탐지한다",
    "작전 명령을 하달하고 임무를 수행한다",
    "병차 전투 장비의 정비 상태를 점검한다",
]
with open("mil_corpus.txt", "w", encoding="utf-8") as fp:
    fp.write("\n".join(mil_sentences * 200))
print("mil_corpus.txt 생성:", os.path.getsize("mil_corpus.txt"), "bytes")


In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
trainer = trainers.BpeTrainer(
    vocab_size=8000,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])

tokenizer.train(files=["mil_corpus.txt"], trainer=trainer)  # 도메인 텍스트
tokenizer.save("mil_bpe.json")

for term in ["대전차화기", "K-9 자주포", "GOP 경계작전"]:
    print(f"{term:12s} -> {tokenizer.encode(term).tokens}")

**실행 결과 예시**

```
대전차화기        -> ['대전차', '화기']
K-9 자주포      -> ['K-9', '자주포']
GOP 경계작전     -> ['GOP', '경계작전']
```

(출력은 코퍼스에 따라 달라지는 예시이다 — 해당 용어가 코퍼스에 충분히 등장해야 이런 병합이 학습되며, 더 자주 등장하면 '대전차화기' 전체가 한 토큰이 되기도 한다.)


## 실습: BERT로 군사 문서 분류

### 🔬 실습 — 한국어 BERT 파인튜닝 (작전 문서 분류)

① **토크나이저와 데이터**: KLUE-BERT 토크나이저로 군사 문장을 토큰화한다.


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")
texts  = ["K2 소총의 정비 주기는 사격 100발 이후이다.",
          "전방 관측소에서 적 병력 이동이 탐지되었다."]
labels = [0, 1]                              # 0: 정비, 1: 정보보고
enc = tokenizer(texts, padding="max_length", truncation=True,
                max_length=64, return_tensors="pt")
print("input_ids:", enc["input_ids"].shape)  # (2, 64)

② **사전학습 분류 모델**: BERT에 분류 head를 붙인다.


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(
    "klue/bert-base", num_labels=2)

③ **파인튜닝 스텝**: [CLS] 표현 기반 분류 손실로 학습.


In [ ]:
torch.manual_seed(0)                          # 분류 헤드 초기화 재현(1장)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
y = torch.tensor(labels)
model.train()
for epoch in range(3):
    out = model(**enc, labels=y)              # 손실 자동 계산
    out.loss.backward(); optimizer.step(); optimizer.zero_grad()
    print(f"ep{epoch+1} loss {out.loss.item():.4f}")

④ **추론**: 새 문장의 클래스 예측.


In [ ]:
model.eval()
test = tokenizer(["야간 정비 절차를 점검하라."], padding="max_length",
                 truncation=True, max_length=64, return_tensors="pt")
with torch.no_grad():
    pred = model(**test).logits.argmax(-1)
print("예측 클래스:", pred.item())            # 0(정비) 기대

**실행 결과 예시**

```
ep1 loss 0.6367
ep2 loss 0.5514
ep3 loss 0.2344
예측 클래스: 0
```

(본문 코드를 시드 0으로 그대로 실행한 결과이다. 두 문장 학습은 코드 흐름을 보이기 위한 장난감 예로, 시드를 바꾸면 분류 헤드의 초기값이 달라져 새 문장을 틀리게 예측하기도 한다.)


---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
